# 03 · Identifiers and delistings: join on CIK, keep the dead companies

Two habits quietly break stock research:

1. **Joining on ticker.** Tickers get renamed, reused and moved between legal entities.
2. **Using today's list of companies.** Companies that went bankrupt or were acquired
   drop out, so the history that's left is the survivors (survivorship bias).

The SEC's CIK (Central Index Key) is permanent for a legal entity, and Varve keeps every
company that ever filed, including the dead ones. The free sample includes some
deliberately awkward cases.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from varve_sample import load

pd.set_option("display.width", 200, "display.max_columns", 20, "display.max_colwidth", 70)
companies = load("fundamentals_companies")
q = load("fundamentals_quarterly")

## One ticker, two companies

In [2]:
companies["tickers"] = companies["tickers_all"].str.split("|")
shared = companies.explode("tickers").groupby("tickers").filter(lambda g: g["cik"].nunique() > 1)
shared.sort_values(["tickers", "first_filed"])[
    ["tickers", "cik", "name", "former_names", "first_filed", "last_filed"]]

,tickers,cik,name,former_names,first_filed,last_filed
14,BBBY,886158,BED BATH & BEYOND INC,NaN,2009-10-07,2023-06-14
20,BBBY,1130713,"BED BATH & BEYOND, INC.","BEYOND, INC.|OVERSTOCK COM INC|OVERSTOCK.COM, INC",2011-07-28,2026-08-04
15,DIS,1001039,WALT DISNEY CO/,DC HOLDCO INC,2009-07-30,2019-02-05
26,DIS,1744489,WALT DISNEY CO,TWDC HOLDCO 613 CORP.,2019-05-08,2026-08-05
21,GOOG,1288776,GOOGLE INC.,GOOGLE INC.,2009-08-04,2015-10-29
25,GOOG,1652044,ALPHABET INC.,NaN,2015-10-29,2026-07-23
21,GOOGL,1288776,GOOGLE INC.,GOOGLE INC.,2009-08-04,2015-10-29
25,GOOGL,1652044,ALPHABET INC.,NaN,2015-10-29,2026-07-23


* **BBBY**: Bed Bath & Beyond (CIK 886158) went bankrupt in 2023. Overstock (CIK
  1130713) bought the brand, took the Bed Bath & Beyond name and has used the BBBY
  symbol too: a different company with different financials under the same name and
  ticker.
* **GOOGL**: Google Inc. (1288776) became a subsidiary of the new Alphabet Inc.
  (1652044) in 2015. **DIS**: likewise for Disney in 2019, during the Fox deal. Each
  reorganization started a new CIK, so a long history spans two.

A ticker join attaches one company's numbers to the other. Even taking only the
"current" holder of each ticker loses the old company's history, or mixes the two.

## Renamed tickers

`tickers_all` lists every symbol recovered for a CIK. Joining on CIK is unaffected when
the symbol changes.

In [3]:
companies[companies["ticker"].isin(["META", "BKNG", "NFLX"])][
    ["cik", "ticker", "tickers_all", "name", "former_names"]]

,cik,ticker,tickers_all,name,former_names
18,1065280,NFLX,NFLX|NLFX,NETFLIX INC,NETFLIX COM INC
19,1075531,BKNG,BKNG|PCLN,BOOKING HOLDINGS INC.,PRICELINE COM INC|PRICELINE GROUP INC.
23,1326801,META,FB|META,"META PLATFORMS, INC.",FACEBOOK INC


Price data you buy elsewhere is usually keyed by ticker and date. Map it to CIK with the
ticker that was in use **on that date**. The full dataset includes a `ticker_history`
table (ticker, CIK, first and last seen) for exactly this.

## Companies that no longer exist

`ticker_current` is empty when the SEC's current ticker list has no symbol for the
company. Their data stays in the dataset.

In [4]:
gone = companies[companies["ticker_current"].isna()].sort_values("last_filed")
gone[["cik", "ticker", "name", "last_filed"]]

,cik,ticker,name,last_filed
21,1288776,GOOGL,GOOGLE INC.,2015-10-29
15,1001039,DIS,WALT DISNEY CO/,2019-02-05
24,1418091,TWTR,"TWITTER, INC.",2022-07-26
12,719739,SIVB,SVB FINANCIAL GROUP,2023-02-24
14,886158,BBBY,BED BATH & BEYOND INC,2023-06-14
3,34088,XOM,EXXON MOBIL CORP,2026-08-03


Most of these are the familiar endings: SVB Financial collapsed in March 2023, Bed
Bath & Beyond filed for bankruptcy a month later, Twitter was taken private in 2022, and
the old Google and Disney entities were replaced by new parents.

Exxon Mobil is the odd one out: it still files, but the SEC's current ticker list now
assigns XOM to a newly registered holding company, ExxonMobil Holdings Corp (CIK
2115436). That's a live example of why `ticker_current` and `last_filed` are both worth
checking before calling a company dead.

## What the last filings looked like

Survivors-only data never shows a company on its way down. Here are SVB's last
quarters: total assets peaked in early 2022 and shrank through the year while quarterly
profit fell. Its last 10-K was filed on 24 February 2023, two weeks before the bank
failed.

In [5]:
svb = q[q["cik"] == 719739].sort_values("period_end")
pd.DataFrame({
    "period_end": svb["period_end"].dt.date,
    "filed": svb["filed"].dt.date,
    "total_assets_B": svb["total_assets"] / 1e9,
    "equity_B": svb["equity"] / 1e9,
    "net_income_M": svb["net_income"] / 1e6,
}).tail(6).round(1)

,period_end,filed,total_assets_B,equity_B,net_income_M
929,2021-09-30,2021-11-08,191.0,14.3,475.0
930,2021-12-31,2022-03-01,211.5,16.2,408.2
931,2022-03-31,2022-05-06,220.4,16.0,533.0
932,2022-06-30,2022-08-08,214.4,15.9,353.0
933,2022-09-30,2022-11-07,212.9,15.5,418.0
934,2022-12-31,2023-02-24,211.8,16.0,305.0


A screen run on today's companies can never select SVB, Bed Bath & Beyond or Twitter,
so a backtest built on it never takes their losses. With point-in-time data, a 2022
screen sees them exactly as investors did.

## Next

**04 · Events**: earnings release times, insider buying and 13D activists, from the
EDGAR daily feed.